# Leçon 12 - Réduction de l'historique de discussion avec Agent Scratchpad

Ce notebook démontre comment gérer le contexte dans de longues conversations en utilisant Microsoft Agent Framework. À mesure que les conversations s'allongent, le nombre de tokens augmente — dépassant finalement la fenêtre de contexte du modèle. Nous abordons cela avec un **modèle de résumé de contexte** et un **agent scratchpad** pour une mémoire persistante.

## Ce que vous allez apprendre :
1. **Pourquoi la gestion du contexte est importante** : Comprendre les limites de tokens et les fenêtres de contexte
2. **Agents sensibles au contexte** : Construire des agents qui gèrent leur propre contexte de conversation
3. **Modèle de résumé du contexte** : Utiliser des outils pour condenser l'historique de la conversation
4. **Agent Scratchpad** : Mémoire persistante qui survit à la réduction du contexte

## Prérequis :
- Configuration Azure OpenAI avec les variables d'environnement configurées
- Compréhension des concepts de base des agents vus dans les leçons précédentes


## Configuration


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Pourquoi la gestion du contexte est importante

Chaque LLM possède une **fenêtre de contexte** finie — le nombre maximum de tokens qu’il peut traiter en une seule requête. Au fur et à mesure qu’une conversation à plusieurs tours progresse :

- **Le nombre de tokens augmente linéairement** avec chaque message de l’utilisateur et chaque réponse de l’assistant.
- **Les tokens de l’invite dominent le coût** car tout l’historique est renvoyé à chaque tour.
- Finalement, la conversation **dépasse la fenêtre de contexte** et le modèle tronque ou génère une erreur.

### Stratégies pour gérer le contexte

| Stratégie | Fonctionnement | Compromis |
|---|---|---|
| **Tronquage** | Supprimer les messages les plus anciens | Perte du contexte initial |
| **Résumé** | Condenser les messages plus anciens en un résumé | Perte de certains détails, mais les points clés sont conservés |
| **Bloc-notes / Mémoire externe** | Stocker les faits clés en dehors de la conversation | Nécessite des appels à des outils, mais survit à toute réduction |

Dans ce notebook, nous combinons le **résumé** avec un **outil de bloc-notes** afin que l’agent puisse maintenir la continuité même lorsque l’historique de la conversation est condensé.


## Création d'un agent contextuel


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Simulation d'une longue conversation

Parcourons une conversation à plusieurs échanges pour voir comment le contexte s'accumule. L'agent doit conserver les détails clés (préférences, budget, dates de voyage) au fil des échanges et démontrer une continuité.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Remarquez comment l'agent conserve le contexte des tours précédents — il connaît le Japon, les sushis, les temples, la photographie, le budget de 3000 $, le voyage en solo et la période d’avril. Dans une conversation courte, cela fonctionne bien, mais à mesure que la conversation s’allonge, il devient coûteux de renvoyer tout l’historique.

Continuons la conversation avec plus de tours pour voir l’accumulation du contexte :


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Modèle de synthèse du contexte

À mesure que la conversation progresse, nous pouvons utiliser un **outil de synthèse** pour condenser le contexte accumulé en un format compact. L’agent appelle cet outil pour enregistrer les préférences clés afin que, même si les messages plus anciens sont supprimés, l’information essentielle soit préservée.

Ce modèle est la base pour une réduction d’historique plus sophistiquée :
1. L’agent identifie les faits clés de la conversation
2. Il appelle l’outil de synthèse pour les enregistrer
3. Les messages plus anciens peuvent être supprimés en toute sécurité car le résumé capture ce qui importe

Ci-dessous, nous définissons un outil `summarize_preferences` que l’agent peut appeler pour enregistrer un résumé compact de ce qu’il a appris.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Résumé

Dans cette leçon, vous avez appris comment gérer le contexte dans les conversations d'agents longue durée en utilisant Microsoft Agent Framework :

### Concepts clés
- **Les fenêtres de contexte sont limitées** — chaque jeton dans l'historique de la conversation coûte de l'argent et compte dans la limite.
- **Les outils de résumé** permettent à l'agent de condenser le contexte accumulé en résumés compacts, réduisant ainsi l'utilisation des jetons tout en préservant les informations essentielles.
- **Les blocs-notes d’agent** fournissent une mémoire externe persistante qui survit à toute réduction de la conversation.

### Ce que vous avez construit
- Un **agent conscient du contexte** qui maintient la continuité sur plusieurs échanges
- Un **outil de résumé** (`summarize_preferences`) qui enregistre les détails clés de l'utilisateur dans un format compact
- Une **conversation multi-échanges** démontrant la rétention du contexte et la gestion des changements

### Applications dans le monde réel
- **Bots de service client** : se souvenir des préférences durant de longues sessions d’assistance
- **Assistants personnels** : suivre les projets en cours sans réexpliquer le contexte
- **Tuteurs éducatifs** : maintenir la progression des élèves à travers de nombreuses interactions

### Prochaines étapes
- Implémenter un outil complet de bloc-notes avec persistance basée sur des fichiers
- Ajouter une troncature automatique de l’historique après résumé
- Combiner avec des bases de données vectorielles pour la recherche sémantique en mémoire
- Construire des agents capables de reprendre des conversations plusieurs jours plus tard avec le contexte complet


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Avertissement** :  
Ce document a été traduit à l’aide du service de traduction par IA [Co-op Translator](https://github.com/Azure/co-op-translator). Bien que nous nous efforcions d’assurer l’exactitude, veuillez noter que les traductions automatisées peuvent contenir des erreurs ou des inexactitudes. Le document original dans sa langue d’origine doit être considéré comme la source faisant foi. Pour des informations critiques, il est recommandé de recourir à une traduction professionnelle humaine. Nous déclinons toute responsabilité en cas de malentendus ou d’interprétations erronées résultant de l’utilisation de cette traduction.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
